# FPGA Audio Equalizer – PYNQ Control Interface

This notebook loads the hardware overlay onto the PYNQ-Z2 and provides
interactive sliders for real-time gain adjustment of the three frequency bands:

| Band | Frequency Range | FFT Bins |
|------|----------------|----------|
| Low  | 0 – 250 Hz     | 0 – 5    |
| Mid  | 250 – 4000 Hz  | 6 – 85   |
| High | 4000 – 24000 Hz| 86 – 511 |

**Prerequisites:** Copy `audio_equalizer.bit` and `audio_equalizer.hwh` to
the same directory as this notebook before running.

In [ ]:
from equalizer_control import AudioEqualizer
import ipywidgets as widgets
from IPython.display import display

In [ ]:
# ── Load overlay ──────────────────────────────────────────────────────────────
eq = AudioEqualizer("audio_equalizer.bit")
eq.start()
print("Overlay loaded:", eq)

In [ ]:
# ── Build interactive UI ──────────────────────────────────────────────────────

style  = {'description_width': '80px'}
layout = widgets.Layout(width='500px')

slider_low = widgets.FloatSlider(
    value=1.0, min=0.0, max=4.0, step=0.05,
    description='Low (bass)',
    continuous_update=True,
    style=style, layout=layout
)

slider_mid = widgets.FloatSlider(
    value=1.0, min=0.0, max=4.0, step=0.05,
    description='Mid',
    continuous_update=True,
    style=style, layout=layout
)

slider_high = widgets.FloatSlider(
    value=1.0, min=0.0, max=4.0, step=0.05,
    description='High (treble)',
    continuous_update=True,
    style=style, layout=layout
)

gain_label = widgets.Label(value='Gains: low=1.00  mid=1.00  high=1.00')

def on_gain_change(change):
    eq.set_gains(
        low=slider_low.value,
        mid=slider_mid.value,
        high=slider_high.value
    )
    gain_label.value = (
        f'Gains: low={slider_low.value:.2f}  '
        f'mid={slider_mid.value:.2f}  '
        f'high={slider_high.value:.2f}'
    )

slider_low.observe(on_gain_change,  names='value')
slider_mid.observe(on_gain_change,  names='value')
slider_high.observe(on_gain_change, names='value')

# ── Preset buttons ────────────────────────────────────────────────────────────
btn_flat    = widgets.Button(description='Flat',         button_style='info')
btn_bass    = widgets.Button(description='Bass Boost',   button_style='success')
btn_treble  = widgets.Button(description='Treble Boost', button_style='warning')
btn_voice   = widgets.Button(description='Voice',        button_style='primary')

def apply_flat(b):
    eq.preset_flat()
    slider_low.value  = 1.0
    slider_mid.value  = 1.0
    slider_high.value = 1.0

def apply_bass(b):
    eq.preset_bass_boost()
    slider_low.value  = 2.0
    slider_mid.value  = 1.0
    slider_high.value = 0.6

def apply_treble(b):
    eq.preset_treble_boost()
    slider_low.value  = 0.6
    slider_mid.value  = 1.0
    slider_high.value = 2.0

def apply_voice(b):
    eq.preset_voice_enhance()
    slider_low.value  = 0.5
    slider_mid.value  = 1.8
    slider_high.value = 1.2

btn_flat.on_click(apply_flat)
btn_bass.on_click(apply_bass)
btn_treble.on_click(apply_treble)
btn_voice.on_click(apply_voice)

presets_row = widgets.HBox([btn_flat, btn_bass, btn_treble, btn_voice])

display(
    widgets.VBox([
        widgets.HTML('<h3>Audio Equalizer Control</h3>'),
        slider_low,
        slider_mid,
        slider_high,
        gain_label,
        widgets.HTML('<b>Presets:</b>'),
        presets_row,
    ])
)

In [ ]:
# ── Stop the core when done ───────────────────────────────────────────────────
# eq.stop()